### Notebook to give an idea how runnables were used to construct standardized components and then how to use them in chains

In [1]:
from abc import ABC, abstractmethod

In [2]:
class Runnable(ABC):
    
    @abstractmethod
    def invoke(input_data):
        pass

In [3]:
import random

In [4]:
class MadeUpLLM(Runnable):

  def __init__(self):
    print('LLM created')

  def invoke(self, prompt):

    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]

    return {'response': random.choice(response_list)}

  def predict(self, prompt):

    response_list = [
        'Delhi is the capital of India',
        'IPL is a cricket league',
        'AI stands for Artificial Intelligence'
    ]

    return {'response': random.choice(response_list)}

In [5]:
llm = MadeUpLLM()

LLM created


In [6]:
class MadeUpPromptTemplate(Runnable):

  def __init__(self, template, input_variables):
    self.template = template
    self.input_variables = input_variables

  def invoke(self, input_dict):
    return self.template.format(**input_dict)
    
  def format(self, input_dict):
    return self.template.format(**input_dict)

In [7]:
template = MadeUpPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

In [8]:
prompt = template.format({'length':'short','topic':'india'})

In [9]:
llm.predict(prompt)

{'response': 'AI stands for Artificial Intelligence'}

### since the components have become standardized, now we'll connect them

In [10]:
class RunnableConnector(Runnable):

  def __init__(self, runnable_list):
    self.runnable_list = runnable_list

  def invoke(self, input_data):

    for runnable in self.runnable_list:
      input_data = runnable.invoke(input_data)

    return input_data

In [11]:
llm = MadeUpLLM()

LLM created


In [12]:
template = MadeUpPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

In [13]:
chain = RunnableConnector([template, llm])

In [14]:
chain.invoke({'length':'long', 'topic':'india'})

{'response': 'Delhi is the capital of India'}

### adding one more component

In [15]:
class MadeUpStrOutputParser(Runnable):

  def __init__(self):
    pass

  def invoke(self, input_data):
    return input_data['response']

In [16]:
llm = MadeUpLLM()

LLM created


In [17]:
template = MadeUpPromptTemplate(
    template='Write a {length} poem about {topic}',
    input_variables=['length', 'topic']
)

In [18]:
parser = MadeUpStrOutputParser()

In [19]:
chain = RunnableConnector([template, llm, parser])

In [20]:
chain.invoke({'length':'long', 'topic':'india'})

'Delhi is the capital of India'

### now we will connect 2 chains to form a bigger chain

In [21]:
llm = MadeUpLLM()

LLM created


In [22]:
template1 = MadeUpPromptTemplate(
    template='Write a joke about {topic}',
    input_variables=['topic']
)

In [23]:
template2 = MadeUpPromptTemplate(
    template='Explain the following joke {response}',
    input_variables=['response']
)

In [24]:
parser = MadeUpStrOutputParser()

In [25]:
chain1 = RunnableConnector([template1, llm])

In [26]:
# chain1.invoke({'topic':'AI'})

In [27]:
chain2 = RunnableConnector([template2, llm, parser])

In [28]:
# chain2.invoke({'response':'This is a joke'})

In [29]:
final_chain = RunnableConnector([chain1, chain2])

In [30]:
final_chain.invoke({'topic':'cricket'})

'AI stands for Artificial Intelligence'